In [ ]:
# Task: Date Alignment

import pandas as pd

# Load news dataset
news_df = pd.read_csv('../data/raw/raw_analyst_ratings.csv') 

# Convert to datetime objects using 'mixed' to handle the different strings in the file
news_df['date'] = pd.to_datetime(news_df['date'], format='mixed', dayfirst=False, errors='coerce', utc=True)

# Convert to naive datetime (removes the +00:00) so it's easier to merge with stock data
news_df['date'] = news_df['date'].dt.tz_localize(None)

# Check the data type
print(f"Column Data Type: {news_df['date'].dtype}")

# Look at the head to see the standardized YYYY-MM-DD format
print(news_df['date'].head(12))

# Check if any dates became 'NaT' (Not a Time)
failed_conversions = news_df['date'].isna().sum()

if failed_conversions == 0:
    print(f"Success! All {len(news_df)} rows were converted to a uniform format.")
else:
    print(f"Warning: {failed_conversions} rows could not be parsed.")


Column Data Type: datetime64[ns]
0    2020-06-05 14:30:54
1    2020-06-03 14:45:20
2    2020-05-26 08:30:07
3    2020-05-22 16:45:06
4    2020-05-22 15:38:59
5    2020-05-22 15:23:25
6    2020-05-22 13:36:20
7    2020-05-22 13:07:04
8    2020-05-22 12:37:59
9    2020-05-22 12:06:17
10   2020-05-22 00:00:00
11   2020-05-22 00:00:00
Name: date, dtype: datetime64[ns]
Success! All 1407328 rows were converted to a uniform format.


In [28]:
# Weekend Alignment Function
def align_to_trading_day(df):
    """
    Shifts Saturday news (+2 days) and Sunday news (+1 day) to Monday.
    This ensures news sentiment can be correlated with market opens.
    """
    # .weekday(): Monday=0, Tuesday=1, ... Saturday=5, Sunday=6
    df['trading_date'] = df['date'].apply(
        lambda x: x + pd.Timedelta(days=2) if x.weekday() == 5 
        else (x + pd.Timedelta(days=1) if x.weekday() == 6 else x)
    )
    # Strip time so we can merge on 'Date' only (YYYY-MM-DD)
    df['trading_date'] = df['trading_date'].dt.date
    return df

news_df = align_to_trading_day(news_df)

# See the results
print("Alignment Verification:")
print(news_df[['date', 'trading_date']].head(12))

# Check for weekends
print("\n--- Count of headlines per day of the week with the weekends added on Monday (0=Mon, 6=Sun) ---")
# This should show 0, 1, 2, 3, 4. If 5 and 6 are 0, it worked!
print(pd.to_datetime(news_df['trading_date']).dt.weekday.value_counts())


Alignment Verification:
                  date trading_date
0  2020-06-05 14:30:54   2020-06-05
1  2020-06-03 14:45:20   2020-06-03
2  2020-05-26 08:30:07   2020-05-26
3  2020-05-22 16:45:06   2020-05-22
4  2020-05-22 15:38:59   2020-05-22
5  2020-05-22 15:23:25   2020-05-22
6  2020-05-22 13:36:20   2020-05-22
7  2020-05-22 13:07:04   2020-05-22
8  2020-05-22 12:37:59   2020-05-22
9  2020-05-22 12:06:17   2020-05-22
10 2020-05-22 00:00:00   2020-05-22
11 2020-05-22 00:00:00   2020-05-22

--- Count of headlines per day of the week with the weekends added on Monday (0=Mon, 6=Sun) ---
trading_date
3    302619
2    300922
1    296505
0    289364
4    217918
Name: count, dtype: int64


In [31]:
# Handling Holidays
from pandas.tseries.holiday import USFederalHolidayCalendar

def align_to_trading_day_with_holidays(df):
    # 1. Identify US Holidays
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['date'].min(), end=df['date'].max())
    
    df['trading_date'] = pd.to_datetime(df['date']).dt.date
    
    # 2. Loop to push dates forward if they hit a weekend or holiday
    def get_next_trading_day(d):
        d = pd.to_datetime(d)
        # While the day is a weekend OR a holiday, add 1 day
        while d.weekday() >= 5 or d in holidays:
            d += pd.Timedelta(days=1)
        return d.date()

    # Note: This takes a moment on 1.4M rows
    print("Aligning dates to next trading day (including holidays)...")
    df['trading_date'] = df['trading_date'].apply(get_next_trading_day)
    return df

news_df = align_to_trading_day_with_holidays(news_df)

# Create a list of some major holidays within your data range
test_holidays = ['2020-01-01', '2019-07-04', '2020-12-25']

# Filter the dataframe to see headlines from those specific days
holiday_check = news_df[news_df['date'].dt.strftime('%Y-%m-%d').isin(test_holidays)]

if not holiday_check.empty:
    print("Holiday Alignment Verification:")
    # Show the original date vs the trading date
    print(holiday_check[['date', 'trading_date']].head(10))
else:
    print("No headlines found on those specific dates, trying a broader search...")


# Alternative check if any trading_date matches a holiday (it shouldn't!)
from pandas.tseries.holiday import USFederalHolidayCalendar
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=news_df['date'].min(), end=news_df['date'].max())
    
    # This should be 0 if the code worked
invalid_dates = news_df[pd.to_datetime(news_df['trading_date']).isin(holidays)]
print(f"Number of headlines still sitting on a holiday: {len(invalid_dates)}")


Aligning dates to next trading day (including holidays)...
Holiday Alignment Verification:
                      date trading_date
42310  2020-01-01 00:00:00   2020-01-02
160573 2020-01-01 00:00:00   2020-01-02
174389 2020-01-01 00:00:00   2020-01-02
185183 2020-01-01 00:00:00   2020-01-02
185518 2020-01-01 15:47:10   2020-01-02
289511 2020-01-01 00:00:00   2020-01-02
380833 2020-01-01 00:00:00   2020-01-02
428226 2020-01-01 00:00:00   2020-01-02
546613 2020-01-01 00:00:00   2020-01-02
555788 2020-01-01 00:00:00   2020-01-02
Number of headlines still sitting on a holiday: 0


In [33]:
# Task: Sentiment Analysis

# Initialize VADER and Progress Tracking

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tqdm import tqdm

# Download the lexicon (the dictionary of 'feeling' words)
nltk.download('vader_lexicon')

# Initialize the analyzer
sia = SentimentIntensityAnalyzer()

# This line allows pandas to use .progress_apply()
tqdm.pandas()

# Run the Sentiment Scoring using the code below
print("Analyzing sentiment for 1.4M headlines...")

# We use 'str(x)' to ensure any non-text headlines don't crash the script
news_df['sentiment_score'] = news_df['headline'].progress_apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\nemsa\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Analyzing sentiment for 1.4M headlines...


100%|██████████| 1407328/1407328 [03:36<00:00, 6497.06it/s]


In [35]:
# Categorize the Scores

def categorize_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

news_df['sentiment_class'] = news_df['sentiment_score'].apply(categorize_sentiment)

# Check the distribution
print("\nSentiment Distribution:")
print(news_df['sentiment_class'].value_counts())


Sentiment Distribution:
sentiment_class
neutral     741194
positive    441858
negative    224276
Name: count, dtype: int64


In [ ]:
# Create the directory
import os
os.makedirs('../data/processed', exist_ok=True)

# Save to CSV
output_path = '../data/processed/aligned_news_sentiment.csv'
news_df.to_csv(output_path, index=False)

print(f"Success! Final dataset saved to {output_path}")

Success! Final dataset saved to ../data/processed/aligned_news_sentiment.csv


In [48]:
# Task: Daily Stock Returns

# First Identify all available stocks in News dataset
available_stocks = news_df['stock'].unique()
print(f"Total unique tickers found in news: {len(available_stocks)}")
print(" ")
print("Sample of tickers:", available_stocks[:20])
print(" ")

# Filter only the five tickers we want
# 1. Define your targets
target_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']

# 2. Get the unique tickers actually present in the CSV
found_tickers = set(news_df['stock'].unique())

# 3. Check which of your targets are present
matches = [t for t in target_stocks if t in found_tickers]
missing = [t for t in target_stocks if t not in found_tickers]

print(f"✅ Found in News: {matches}")
if missing:
    print(f"❌ Missing from News: {missing}")
    print(" ")
else:
    print("🚀 All target stocks were successfully located in the dataset!")
print(" ")
# Filter the dataframe
filtered_news = news_df[news_df['stock'].isin(target_stocks)].copy()

print(f"Rows remaining after filtering: {len(filtered_news)}")

# Filtering and grouping to calculate the daily average sentiment for each stock indvidually
# Filter using the 'matches' list we just validated
filtered_news = news_df[news_df['stock'].isin(matches)].copy()

# Group by Stock and Date to get the Daily Average Sentiment
daily_sentiment = filtered_news.groupby(['stock', 'trading_date'])['sentiment_score'].mean().reset_index()

# Rename the column to be clear
daily_sentiment = daily_sentiment.rename(columns={'sentiment_score': 'avg_sentiment'})

print("\nFinal Aggregated Shape:", daily_sentiment.shape)
print("\n--- Daily Aggregated Sentiment Preview ---")
print(daily_sentiment.head(5))
print(" ")

# Check how many 'Trading Days' of news we have for each of the stocks
print("Days of news coverage per stock:")
print(daily_sentiment['stock'].value_counts())

Total unique tickers found in news: 6204
 
Sample of tickers: ['A' 'AA' 'AAC' 'AADR' 'AAL' 'AAMC' 'AAME' 'AAN' 'AAOI' 'AAON' 'AAP'
 'AAPL' 'AAU' 'AAV' 'AAVL' 'AAWW' 'AAXJ' 'AB' 'ABAC' 'ABAX']
 
✅ Found in News: ['AAPL', 'AMZN', 'GOOG', 'NVDA']
❌ Missing from News: ['META']
 
 
Rows remaining after filtering: 5064

Final Aggregated Shape: (1577, 3)

--- Daily Aggregated Sentiment Preview ---
  stock trading_date  avg_sentiment
0  AAPL   2020-03-09      -0.302067
1  AAPL   2020-03-10      -0.090787
2  AAPL   2020-03-11      -0.023850
3  AAPL   2020-03-12      -0.078360
4  AAPL   2020-03-13      -0.059727
 
Days of news coverage per stock:
stock
NVDA    1136
GOOG     352
AAPL      61
AMZN      28
Name: count, dtype: int64


In [49]:
# Second Loading datasets into Python as DataFrames and then calculating Daily Stock Returns using .pct_change()

import pandas as pd

# List of the 4 tickers found in the news data
tickers = ['AAPL', 'AMZN', 'GOOG', 'NVDA'] 
stock_frames = []

for ticker in tickers:
    # 1. Load the CSV
    file_path = f'../data/raw/{ticker}.csv' 
    df = pd.read_csv(file_path)
    
    # 2. Basic Cleaning
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df.sort_values('Date')
    
    # 3. Calculate Daily Return ( (Close_t / Close_t-1) - 1 ) * 100
    df['Daily_Return'] = df['Close'].pct_change() * 100
    
    # 4. Add a Ticker column so we can distinguish them after merging
    df['Ticker'] = ticker
    
    # 5. Keep only the columns we need to save memory
    df = df[['Date', 'Ticker', 'Close', 'Daily_Return']]
    
    # Drop the first row of each stock (which has NaN for Daily_Return)
    df = df.dropna()
    
    stock_frames.append(df)

# 2. Combine all stocks into one master table
all_stock_prices = pd.concat(stock_frames, ignore_index=True)

print("Master Price DataFrame created!")

# Previewing each stock's calculated returns
for df in stock_frames:
    ticker = df['Ticker'].iloc[0] # Get the ticker name from the first row
    print(f"\n--- {ticker} Preview ---")
    print(df.head(3))

Master Price DataFrame created!

--- AAPL Preview ---
        Date Ticker     Close  Daily_Return
1 2009-01-05   AAPL  2.836553      4.220416
2 2009-01-06   AAPL  2.789767     -1.649399
3 2009-01-07   AAPL  2.729484     -2.160860

--- AMZN Preview ---
        Date Ticker  Close  Daily_Return
1 2009-01-05   AMZN  2.703     -0.551871
2 2009-01-06   AMZN  2.868      6.104327
3 2009-01-07   AMZN  2.810     -2.022318

--- GOOG Preview ---
        Date Ticker     Close  Daily_Return
1 2009-01-05   GOOG  8.115089      2.094467
2 2009-01-06   GOOG  8.263762      1.832057
3 2009-01-07   GOOG  7.965677     -3.607137

--- NVDA Preview ---
        Date Ticker     Close  Daily_Return
1 2009-01-05   NVDA  0.203319      1.836972
2 2009-01-06   NVDA  0.210196      3.382205
3 2009-01-07   NVDA  0.197589     -5.997831


In [53]:
# Task: Aggreagate and Correlate

# Merging with News data
# Ensure the sentiment dates match the stock dates format
daily_sentiment['trading_date'] = pd.to_datetime(daily_sentiment['trading_date'])

# The final merge
merged_final = pd.merge(
    all_stock_prices, 
    daily_sentiment, 
    left_on=['Date', 'Ticker'], 
    right_on=['trading_date', 'stock'], 
    how='inner'
)

# Clean up
merged_final.drop(columns=['trading_date', 'stock'], inplace=True)

print(f"Final dataset prepared with {len(merged_final)} rows.")

# Show a sample of rows from different stocks to ensure variety
print("Merged Data Preview (Random Sample):")
print(merged_final.sample(10).sort_values(['Ticker', 'Date']))

# Check how many data points we have for each stock after the merge
print("\nData points per stock after merging:")
print(merged_final['Ticker'].value_counts())

Final dataset prepared with 1575 rows.
Merged Data Preview (Random Sample):
           Date Ticker      Close  Daily_Return  avg_sentiment
48   2020-05-18   AAPL  76.379913      2.356121       0.056575
96   2018-11-26   GOOG  52.074638      2.416306       0.201400
97   2018-11-27   GOOG  51.865570     -0.401478       0.279400
345  2020-01-22   GOOG  73.792526      0.104419      -0.034233
834  2016-01-20   NVDA   0.670345      0.548871       0.296000
850  2016-03-29   NVDA   0.866446      1.607796       0.170200
934  2016-12-14   NVDA   2.375413      5.791394       0.116033
1046 2017-07-24   NVDA   4.101495     -1.160012       0.000000
1184 2018-04-19   NVDA   5.666393     -3.101082      -0.438600
1447 2019-10-31   NVDA   4.999741     -0.975379       0.000000

Data points per stock after merging:
Ticker
NVDA    1135
GOOG     351
AAPL      61
AMZN      28
Name: count, dtype: int64
